# CountriesCount

[Codility task](https://app.codility.com/programmers/trainings/7/countries_count/) - Exercise 7, Data Structures - *hard*

A map is an `N x M` matrix `A` of colors. Two cells belong to the same **country** if they have the same color and one can walk from one to the other moving only north, south, east or west over cells of that color.

`solution(A)` returns the number of countries.

Example: the 7 x 3 matrix below has 11 countries.

```
5 4 4
4 3 4
3 2 4
2 2 2
3 3 4
1 4 4
4 1 1
```

Assumptions: `N`, `M` in `[1..300,000]`, `N * M <= 300,000`, values in `[-10^9..10^9]`.

## Idea: connected components

Countries are the connected components of the grid graph whose edges join orthogonal neighbours with equal values.

1. Scan the cells in order.
2. At an unvisited cell, count a new country and flood-fill its component with an **explicit stack** (a recursive DFS would blow the stack: one country can have 300,000 cells).
3. `visited` is a flat `bytearray` of size `N * M`, which also works for shapes like `1 x 300000` or `300000 x 1`.

Time O(N * M), space O(N * M).

In [ ]:
def solution(A):
    n, m = len(A), len(A[0])
    visited = bytearray(n * m)
    countries = 0

    for si in range(n):
        for sj in range(m):
            if visited[si * m + sj]:
                continue
            countries += 1
            color = A[si][sj]
            visited[si * m + sj] = 1
            stack = [(si, sj)]
            while stack:
                i, j = stack.pop()
                for ni, nj in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)):
                    if 0 <= ni < n and 0 <= nj < m:
                        k = ni * m + nj
                        if not visited[k] and A[ni][nj] == color:
                            visited[k] = 1
                            stack.append((ni, nj))
    return countries

## Official example

In [ ]:
A = [
    [5, 4, 4],
    [4, 3, 4],
    [3, 2, 4],
    [2, 2, 2],
    [3, 3, 4],
    [1, 4, 4],
    [4, 1, 1],
]
assert solution(A) == 11
print("example OK")

## Edge cases

In [ ]:
assert solution([[7]]) == 1                                  # 1 x 1
assert solution([[1, 1, 1]]) == 1                            # one row, one country
assert solution([[1], [1], [1]]) == 1                        # one column
assert solution([[1, 2, 1]]) == 3                            # same colour but not connected
assert solution([[1, 2], [2, 1]]) == 4                       # diagonal neighbours are NOT connected
assert solution([[-10**9, 10**9], [10**9, -10**9]]) == 4     # extreme values
assert solution([[3] * 5 for _ in range(4)]) == 1            # everything the same
checker = [[(i + j) % 2 for j in range(6)] for i in range(5)]
assert solution(checker) == 30                               # checkerboard: every cell is a country
stripes = [[i % 2] * 4 for i in range(6)]
assert solution(stripes) == 6                                # alternating horizontal stripes
print("edge cases OK")

## Cross-check with union-find

An independent algorithm for the same problem: **union-find** (disjoint sets). Every cell starts as its own set; merge it with its right and lower neighbours when the colours match; the number of countries is the number of distinct roots.

In [ ]:
import random


def union_find_count(A):
    n, m = len(A), len(A[0])
    parent = list(range(n * m))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]       # path halving
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for i in range(n):
        for j in range(m):
            if j + 1 < m and A[i][j] == A[i][j + 1]:
                union(i * m + j, i * m + j + 1)
            if i + 1 < n and A[i][j] == A[i + 1][j]:
                union(i * m + j, (i + 1) * m + j)
    return sum(1 for x in range(n * m) if find(x) == x)


assert union_find_count(A) == 11

rng = random.Random(9)
for _ in range(3000):
    n, m = rng.randint(1, 8), rng.randint(1, 8)
    colors = rng.choice([1, 2, 3, 5])
    grid = [[rng.randint(-1, colors - 2) for _ in range(m)] for _ in range(n)]
    assert solution(grid) == union_find_count(grid), grid
print("flood fill and union-find agree on 3000 random maps")

## Large inputs

In [ ]:
import time

rng = random.Random(4)


def timed(name, grid):
    start = time.perf_counter()
    result = solution(grid)
    print(f"{name:28s} -> {result:>7} countries in {time.perf_counter() - start:.3f}s")
    return result


assert timed("1 x 300000 alternating", [[i % 2 for i in range(300_000)]]) == 300_000
assert timed("300000 x 1 same colour", [[5] for _ in range(300_000)]) == 1
assert timed("547 x 548 same colour", [[0] * 548 for _ in range(547)]) == 1
assert timed("547 x 548 checkerboard", [[(i + j) % 2 for j in range(548)] for i in range(547)]) == 547 * 548

random_grid = [[rng.randint(0, 1) for _ in range(548)] for _ in range(547)]
assert timed("547 x 548 random, 2 colours", random_grid) == union_find_count(random_grid)

snake = [[0 if (i % 2 == 0 or (i % 4 == 1 and j == 599) or (i % 4 == 3 and j == 0)) else 1
          for j in range(600)] for i in range(500)]
assert timed("500 x 600 snake (long country)", snake) == 251   # one long 0-country + 250 rows of 1s

## Notes

- Diagonal neighbours are **not** adjacent (see the 2 x 2 checkerboard test).
- A recursive DFS would hit Python's recursion limit on a single large country; the explicit stack does not.
- Two shapes to remember: very tall or very wide matrices, where `N` or `M` is 300,000 and the other is 1. A flat index `i * m + j` keeps `visited` small and simple.
- Union-find is a good alternative when you also need to answer "are these two cells in the same country?" repeatedly.